In [1]:
# 처음 한 번만
#%pip install -q google-genai python-dotenv pillow

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os

from dotenv import load_dotenv
from google import genai
from google.genai import types

api_key = os.environ.get("GEMINI_API_KEY")
MODEL = "gemini-3.1-flash-lite"
client = genai.Client(api_key=api_key, http_options=types.HttpOptions(timeout=60_000))

# 영수증 이미지 생성(영수증 이미지가 없어서)

In [12]:
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont

def generate_receipt_image(file_name, items, store_name, date_time, address):
    FONT_CANDIDATES = [
        "C:/Windows/Fonts/malgun.ttf",                               # Windows 맑은 고딕
        "/System/Library/Fonts/AppleSDGothicNeo.ttc",                # macOS
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",           # Linux 나눔고딕
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",    # Linux Noto
    ]
    font_path = next(p for p in FONT_CANDIDATES if Path(p).exists())
    big, font = ImageFont.truetype(font_path, 28), ImageFont.truetype(font_path, 20)
    
    img = Image.new("RGB", (480, 520), "white")
    draw = ImageDraw.Draw(img)
    y = 30
    draw.text((240, y), store_name, font=big, fill="black", anchor="mt"); y += 50
    draw.text((30, y), date_time, font=font, fill="black"); y += 30
    draw.text((30, y), address, font=font, fill="black"); y += 40
    draw.line((30, y, 450, y), fill="black", width=2); y += 20
    for name, qty, price in items:
        draw.text((30, y), name, font=font, fill="black")
        draw.text((280, y), f"x{qty}", font=font, fill="black")
        draw.text((450, y), f"{price:,}", font=font, fill="black", anchor="ra")
        y += 36
    draw.line((30, y, 450, y), fill="black", width=2); y += 20
    draw.text((30, y), "합계", font=big, fill="black")
    draw.text((450, y), f"{sum(p for _, _, p in ITEMS):,}원", font=big, fill="black", anchor="ra")
    
    img = img.rotate(3, expand=True, fillcolor="white")             # 살짝 기울여서 '사진'처럼
    img.save("receipt_sample.png")

    return img
IMAGE = "receipt_sample.png"
# ITEMS = [("카페라떼", 1, 5000), ("치즈케이크", 1, 6500)]   # (이름, 수량, 금액)    
# img = generate_receipt_image(IMAGE, ITEMS, "코드브릿지 카페1", "2026-11-13 13:24", "서울시 가상구 연습로 121")

# 이미지 설명 받기

In [15]:
from PIL import Image
IMAGE = "receipt_sample.png"
# IMAGE = "dog.jpeg"
image = Image.open(IMAGE)

res = client.models.generate_content(
    model=MODEL,
    contents=[image, "이 이미지에 무엇이 있는지 3문장으로 설명해 주세요."],
    config=types.GenerateContentConfig(max_output_tokens=300),
)
print(res.text)
print(f"\n토큰: 입력 {res.usage_metadata.prompt_token_count} / 출력 {res.usage_metadata.candidates_token_count}")

이 이미지는 '코드브릿지 카페1'에서 발행한 영수증을 보여주고 있습니다. 영수증에는 2026년 11월 13일 오후 1시 24분에 서울시 가상구 연습로 121에서 발행된 내역이 기재되어 있습니다. 주문한 품목인 카페라떼와 치즈케이크의 가격을 합산하여 총 11,500원이라는 금액을 나타내고 있습니다.

토큰: 입력 1104 / 출력 106


In [16]:
from pydantic import BaseModel, Field


class Item(BaseModel):
    name: str = Field(description="품목 이름")
    quantity: int = Field(description="수량")
    price: int = Field(description="이 품목의 금액(원). 쉼표 없이 숫자만")


class Receipt(BaseModel):
    store: str | None = Field(description="가게 이름. 안 보이면 null")
    date: str | None = Field(description="YYYY-MM-DD 형식. 안 보이면 null")
    items: list[Item]
    total: int | None = Field(description="영수증에 적힌 합계(원). 안 보이면 null")


INSTRUCTIONS = """영수증 이미지에서 정보를 읽어 지정된 형식으로만 답합니다.
- 이미지에 보이는 내용만 적습니다. 보이지 않거나 흐린 값은 추측하지 말고 null로 둡니다.
- 카드번호, 승인번호, 전화번호는 적지 않습니다."""

res = client.models.generate_content(
    model=MODEL,
    contents=[image, "이 영수증을 읽어 주세요."],
    config=types.GenerateContentConfig(
        system_instruction=INSTRUCTIONS,
        response_mime_type="application/json",     # JSON으로 답하게
        response_schema=Receipt,                   # 이 형식으로
        max_output_tokens=800,
    ),
)
r: Receipt = res.parsed
print(r.model_dump_json(indent=2))

{
  "store": "코드브릿지 카페1",
  "date": "2026-11-13",
  "items": [
    {
      "name": "카페라떼",
      "quantity": 1,
      "price": 5000
    },
    {
      "name": "치즈케이크",
      "quantity": 1,
      "price": 6500
    }
  ],
  "total": 11500
}
